# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Answer (plain words):**

- **One row =** one piece of content (`content_hash_id`), belonging to one client (`client_hash_id`), on one calendar day (`report_date`). Grain: client x content x day.
- **Table(s):** `fact_content_daily_performance` only.
- **Time window (dev month):** `month = '2026-03'` (mid-panel). The final month `2026-06` is the sealed test month (matches the `_sample` table) and is never used for label/feature logic here.
- **Predict / rank (label or proxy):** binary label `has_ai_referral` = whether the content received at least one AI-referral session that day (`sessions_ai > 0`). This is a proxy for "is this content currently visible to AI answer engines (ChatGPT, Perplexity, Gemini, Copilot, Claude, Meta)."
- **Deliberately excluded:** rows where `client_has_ga4 = FALSE`. AI-referral sessions are a GA4-derived signal, so a client with no GA4 connection can never produce a true positive -- keeping those rows would just add uninformative zeros and silently bias the label toward "no AI referral" for reasons that have nothing to do with the content itself.


In [12]:
import duckdb
from google.colab import userdata

token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
table_path = f"{rel}/fact_content_daily_performance/**/*.parquet"

DEV_MONTH = '2026-03'  # mid-panel month; final month 2026-06 is sealed test, never used here

# quick sanity peek -- confirms the table loads and the dev month exists
con.sql(f"""
    SELECT month, COUNT(*) AS rows
    FROM read_parquet('{table_path}')
    WHERE month = '{DEV_MONTH}'
    GROUP BY month
""").show()


┌─────────┬─────────┐
│  month  │  rows   │
│ varchar │  int64  │
├─────────┼─────────┤
│ 2026-03 │ 9841378 │
└─────────┴─────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Answer (plain words):**

- **Label:** `sessions_ai` -> derived binary `has_ai_referral` = `sessions_ai > 0`.
- **Features (candidates, same-day GSC/GA4 snapshot):** `gsc_impressions`, `gsc_avg_position`, `ga4_engaged_sessions`, `scroll_events`, `sessions_organic`.
- **Context (not fed to a model, used for filtering/grouping only):** `client_hash_id`, `content_hash_id`, `report_date`, `month`, `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`.
- **Excluded, with why:**
  - `ga4_sessions`, `ga4_pageviews`, `ga4_total_engagement_sec` -> excluded from the feature set because they are aggregates that structurally contain `sessions_ai` as a component (a session that came from AI referral is still counted inside total sessions/pageviews). Using them risks partial leakage even though they aren't the label itself.
  - `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` -> excluded entirely. These are just the per-engine breakdown of the label (`sessions_ai` = sum of these). Using any of them as a feature is the same mistake as using the label.


In [13]:
# No query needed here -- this section is the bucket sort itself (see markdown above).
# The one thing worth checking mechanically: confirm sessions_ai really does equal
# the sum of the per-engine AI columns, which is *why* those columns are excluded.

con.sql(f"""
    SELECT
        SUM(sessions_ai) AS total_sessions_ai,
        SUM(ai_chatgpt + ai_perplexity + ai_gemini + ai_copilot + ai_claude + ai_meta + ai_other) AS sum_of_engines
    FROM read_parquet('{table_path}')
    WHERE month = '{DEV_MONTH}'
""").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────┬────────────────┐
│ total_sessions_ai │ sum_of_engines │
│      int128       │     int128     │
├───────────────────┼────────────────┤
│              8911 │           8914 │
└───────────────────┴────────────────┘



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Verification queries (grain, counts, availability) + 5-feature frame + the deliberate leak.**

All queries below run on `month = '2026-03'`, filtered to `client_has_ga4 = TRUE` (per the exclusion rule from Section 1), since GA4 is required for the AI-referral label to be meaningful.


In [14]:
# ---- Query 1: GRAIN CHECK ----
# One row really should be one (client, content, day). Total rows vs distinct combos must match.
grain = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS distinct_grain
    FROM read_parquet('{table_path}')
    WHERE month = '{DEV_MONTH}'
      AND client_has_ga4 = TRUE
""").df()
print("Query 1 -- grain check (total_rows should equal distinct_grain):")
print(grain)

# ---- Query 2: SLICE ROW COUNT + DATE SPAN ----
span = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{table_path}')
    WHERE month = '{DEV_MONTH}'
      AND client_has_ga4 = TRUE
""").df()
print("\nQuery 2 -- slice row count + date span:")
print(span)

# ---- Query 3: AVAILABILITY, using IS TRUE ----
availability = con.sql(f"""
    SELECT
        COUNT(*) AS rows_before_filter,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_surviving_ga4_available
    FROM read_parquet('{table_path}')
    WHERE month = '{DEV_MONTH}'
      AND client_has_ga4 = TRUE
""").df()
print("\nQuery 3 -- availability check (IS TRUE):")
print(availability)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 -- grain check (total_rows should equal distinct_grain):
   total_rows  distinct_grain
0     6822637         6822637

Query 2 -- slice row count + date span:
   row_count   min_date   max_date
0    6822637 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Query 3 -- availability check (IS TRUE):
   rows_before_filter  rows_surviving_ga4_available
0             6822637                        413966


In [16]:
# ---- Build the working frame for feature engineering (grain-checked + availability-checked) ----
df = con.sql(f"""    SELECT        client_hash_id, content_hash_id, report_date,        gsc_impressions, gsc_avg_position, ga4_engaged_sessions,        scroll_events, sessions_organic,        sessions_ai    FROM read_parquet('{table_path}')    WHERE month = '{DEV_MONTH}'      AND client_has_ga4 = TRUE      AND ga4_data_available IS TRUE""").df()
# Label: was this content getting any AI-referral traffic that day?
# Some rows have GSC unavailable for that client/day even though GA4 is present,
# which leaves gsc_impressions / gsc_avg_position as NaN. Missing GSC data here means
# "no search signal recorded," so we fill with 0 rather than drop the row.
gsc_cols = ['gsc_impressions', 'gsc_avg_position']
df[gsc_cols] = df[gsc_cols].fillna(0)

df['has_ai_referral'] = (df['sessions_ai'] > 0).astype(int)
print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(413966, 10)


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_avg_position,ga4_engaged_sessions,scroll_events,sessions_organic,sessions_ai,has_ai_referral
0,client_65de48885f4ef01b,content_09be8cc7fcb222af,2026-03-01,0,0.0,0,0,0,0,0
1,client_65de48885f4ef01b,content_851afac9fe13612e,2026-03-01,0,0.0,0,0,0,0,0
2,client_65de48885f4ef01b,content_cee6c6fc8c51af14,2026-03-01,0,0.0,0,0,0,0,0
3,client_65de48885f4ef01b,content_5e120e972f11f833,2026-03-01,0,0.0,0,0,0,0,0
4,client_65de48885f4ef01b,content_16a7291bb6ecaebe,2026-03-01,0,0.0,0,0,0,0,0


In [17]:
# ---- 5 FEATURES (max), each "knowable at the decision moment because..." ----

feature_frame = df[[
    'client_hash_id', 'content_hash_id', 'report_date',
    'gsc_impressions', 'gsc_avg_position', 'ga4_engaged_sessions',
    'scroll_events', 'sessions_organic',
    'has_ai_referral'
]].copy()

feature_notes = {
    'gsc_impressions':     "knowable at the decision moment because it comes straight from that day's Search Console pull -- it's a search-visibility count, not an AI-referral outcome.",
    'gsc_avg_position':    "knowable at the decision moment because it's the day's average ranking position from GSC, computed independently of any AI engine's referral traffic.",
    'ga4_engaged_sessions':"knowable at the decision moment because it's GA4's own engagement count for that day, aggregated across all channels -- it doesn't require knowing the AI-referral split first.",
    'scroll_events':       "knowable at the decision moment because it's a raw GA4 interaction event captured on-page, unrelated to which channel brought the visitor.",
    'sessions_organic':    "knowable at the decision moment because it's the GA4 organic-channel session count, a separate bucket from the sessions_ai bucket used in the label.",
}
for k, v in feature_notes.items():
    print(f"- {k}: {v}")

feature_frame.head()


- gsc_impressions: knowable at the decision moment because it comes straight from that day's Search Console pull -- it's a search-visibility count, not an AI-referral outcome.
- gsc_avg_position: knowable at the decision moment because it's the day's average ranking position from GSC, computed independently of any AI engine's referral traffic.
- ga4_engaged_sessions: knowable at the decision moment because it's GA4's own engagement count for that day, aggregated across all channels -- it doesn't require knowing the AI-referral split first.
- scroll_events: knowable at the decision moment because it's a raw GA4 interaction event captured on-page, unrelated to which channel brought the visitor.
- sessions_organic: knowable at the decision moment because it's the GA4 organic-channel session count, a separate bucket from the sessions_ai bucket used in the label.


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_avg_position,ga4_engaged_sessions,scroll_events,sessions_organic,has_ai_referral
0,client_65de48885f4ef01b,content_09be8cc7fcb222af,2026-03-01,0,0.0,0,0,0,0
1,client_65de48885f4ef01b,content_851afac9fe13612e,2026-03-01,0,0.0,0,0,0,0
2,client_65de48885f4ef01b,content_cee6c6fc8c51af14,2026-03-01,0,0.0,0,0,0,0
3,client_65de48885f4ef01b,content_5e120e972f11f833,2026-03-01,0,0.0,0,0,0,0
4,client_65de48885f4ef01b,content_16a7291bb6ecaebe,2026-03-01,0,0.0,0,0,0,0


In [18]:
# ---- THE TRAP: add ONE label-derived column on purpose ----
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

leaky_frame = feature_frame.copy()

# This column is derived directly from the label (sessions_ai), not from independent evidence.
leaky_frame['sessions_ai_log'] = np.log1p(df['sessions_ai'])

X_leaky = leaky_frame[['gsc_impressions', 'gsc_avg_position', 'ga4_engaged_sessions',
                       'scroll_events', 'sessions_organic', 'sessions_ai_log']]
y = leaky_frame['has_ai_referral']

Xtr, Xte, ytr, yte = train_test_split(X_leaky, y, test_size=0.3, random_state=42, stratify=y)
model_leaky = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
auc_leaky = roc_auc_score(yte, model_leaky.predict_proba(Xte)[:, 1])
print(f"AUC WITH leaked column (sessions_ai_log): {auc_leaky:.4f}  <- suspiciously close to perfect")

# ---- Remove the leak, keep the honest number ----
X_honest = feature_frame[['gsc_impressions', 'gsc_avg_position', 'ga4_engaged_sessions',
                          'scroll_events', 'sessions_organic']]

Xtr2, Xte2, ytr2, yte2 = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
model_honest = LogisticRegression(max_iter=1000).fit(Xtr2, ytr2)
auc_honest = roc_auc_score(yte2, model_honest.predict_proba(Xte2)[:, 1])
print(f"AUC WITHOUT leaked column (honest):        {auc_honest:.4f}  <- the real, decision-time-safe number")


AUC WITH leaked column (sessions_ai_log): 1.0000  <- suspiciously close to perfect
AUC WITHOUT leaked column (honest):        0.6425  <- the real, decision-time-safe number


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Answer (plain words) -- one named limitation:**

This slice can never tell us *why* a content piece got AI-referral traffic -- only *that* it did. The dataset has no query-level or prompt-level detail (no record of what someone actually asked ChatGPT/Perplexity/etc. that led to the referral), so `has_ai_referral` is an observed, directional signal of AI-engine visibility, not a causal or diagnostic one. Two supporting notes:

- Coverage is uneven across clients: `client_has_gsc` / `client_has_ga4` differ per client, and early history may be GSC-only for some clients -- so cross-client comparisons of AI-referral rate can partly reflect measurement coverage, not real differences in visibility.
- All numbers here are decision-support / directional, not confirmed causal effects.


In [19]:
# Quick check supporting the limitation above: how many client-days have partial coverage
# (GSC only, no GA4, or vice versa) within the dev month.
coverage = con.sql(f"""
    SELECT
        client_has_gsc, client_has_ga4,
        COUNT(*) AS rows
    FROM read_parquet('{table_path}')
    WHERE month = '{DEV_MONTH}'
    GROUP BY client_has_gsc, client_has_ga4
    ORDER BY rows DESC
""").df()
print(coverage)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   client_has_gsc  client_has_ga4     rows
0            True            True  6822637
1            True           False  3018741


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.